In [21]:
import os
import numpy as np
import pandas as pd
import librosa
import noisereduce as nr

from scipy import signal
from tqdm import tqdm
import skimage

In [22]:
class SpectrogramAndMFCCExtraction:
    """
    Extract mel spectrograms and MFCC features
    from segmented audio recordings.
    """

    def __init__(
        self,
        path_audio_files,
        path_segment_data,
        path_save_spec_arrays,
        cutoff,
        nyq_fr,
        **kwargs
    ):
        """
        Parameters
        ----------
        path_audio_files : str
            Directory containing audio files.

        path_segment_data : str
            CSV file containing segment annotations.

        path_save_spec_arrays : str
            Directory where spectrogram arrays will be saved.

        cutoff : float
            Low-pass filter cutoff frequency.

        nyq_fr : float
            Nyquist frequency.
        """

        self.path_audio_files = path_audio_files
        self.path_segment_data = path_segment_data
        self.path_save_spec_arrays = path_save_spec_arrays
        self.cutoff = cutoff
        self.nyq_fr = nyq_fr
        self.kwargs = kwargs

        # Create output directory if it does not exist
        os.makedirs(
            self.path_save_spec_arrays,
            exist_ok=True
        )

    def read_audio_file(self, audio_file):
        """
        Read an audio file.
        Parameters
        ----------
        audio_file : str, Audio filename.
        Returns
        -------
        np.ndarray, Audio signal.
        """
        full_path = os.path.join(self.path_audio_files,audio_file)
        signal_data, _ = librosa.load(full_path,sr=self.kwargs["sr"])
        return signal_data

    def reduce_noise(self, signal_data):
        """
        Reduce background noise.
        Parameters
        ----------
        signal_data : np.ndarray, Audio signal.
        Returns
        -------
        np.ndarray, Noise-reduced signal.
        """
        reduced_noise = nr.reduce_noise(y=signal_data,sr=self.kwargs["sr"])
        return reduced_noise

    @staticmethod
    def butter_lowpass(cutoff, nyq_freq, order=4):
        """
        Create Butterworth low-pass filter coefficients.
        """
        normal_cutoff = float(cutoff) / nyq_freq
        b, a = signal.butter(order, normal_cutoff, btype="lowpass")
        return b, a

    def butter_lowpass_filter(self, data, cutoff_freq, nyq_freq, order=4):
        """
        Apply a low-pass Butterworth filter.
        Parameters
        ----------
        data : np.ndarray, Audio signal.
        cutoff_freq : float, Cutoff frequency.
        nyq_freq : float, Nyquist frequency.
        order : int, Filter order.
        Returns
        -------
        np.ndarray, Filtered signal.
        """
        b, a = self.butter_lowpass(cutoff_freq, nyq_freq, order=order)
        filtered_signal = signal.filtfilt(b, a, data)
        return filtered_signal

    def preprocess_audio_file(self, audio_file):
        """
        Complete preprocessing pipeline.
        Steps
        -----
        1. Read audio
        2. Reduce noise
        3. Apply low-pass filter
        Returns
        -------
        np.ndarray
            Preprocessed signal.
        """
        signal_data = self.read_audio_file(audio_file)
        reduced_noise = self.reduce_noise(signal_data)
        filtered_signal = self.butter_lowpass_filter(
            reduced_noise,
            self.cutoff,
            self.nyq_fr
        )
        return filtered_signal

    def read_segment_data(self):
        """
        Read segment annotation CSV.
        Returns
        -------
        list
            List of tuples:
            (audio, segment, start, end)
        """
        df = pd.read_csv(self.path_segment_data)
        required_columns = [
            "Segment",
            "Start",
            "End",
            "Audio"
        ]
        missing_cols = [
            col for col in required_columns
            if col not in df.columns
        ]
        if missing_cols:
            raise ValueError(
                f"Missing columns: {missing_cols}"
            )
        audio_segment_st_end = list(
            zip(
                df["Audio"],
                df["Segment"],
                df["Start"],
                df["End"]
            )
        )

        return audio_segment_st_end

    def audio2_segment_indexes(self):
        """
        Map audio files to their segments.
        Returns
        -------
        dict
            {
                audio_file:
                [(segment, start, end), ...]
            }
        """
        audio_segment_st_end = (self.read_segment_data())
        audio2data = {}
        for audio, segment, st, end in (audio_segment_st_end):
            if audio not in audio2data:
                audio2data[audio] = []
            audio2data[audio].append((segment, int(st), int(end)))
        return audio2data

    @staticmethod
    def audio_frame_to_mel_spectrogram(audio_frame, **kwargs):
        """
        Extract mel spectrogram.
        Parameters
        ----------
        audio_frame : np.ndarray, Audio segment.
        Returns
        -------
        np.ndarray, Mel spectrogram.
        """

        mel_spectrogram = (
            librosa.feature.melspectrogram(
                y=audio_frame.astype(np.float32),
                **kwargs
            )
        )

        return mel_spectrogram

    def segment_mfccs(self, segment_signal):
        """
        Extract MFCC statistics.
        Parameters
        ----------
        segment_signal : np.ndarray, Audio segment.
        Returns
        -------
        np.ndarray, Concatenated MFCC means and standard deviations.
        """
        mfccs = librosa.feature.mfcc(
            y=segment_signal,
            sr=self.kwargs["sr"],
            n_mfcc=20,
            n_mels=128,
            fmin=4000,
            fmax=8000,
            n_fft=1024,
            hop_length=256
        )

        means = mfccs.mean(axis=1)
        stdvs = mfccs.std(axis=1)
        features = np.hstack([means, stdvs])
        return features

    def Execution(self):
        """
        Run spectrogram and MFCC extraction.
        Parameters
        ----------
        **kwargs :
            Additional parameters for
            librosa.feature.melspectrogram()
        Returns
        -------
        pd.DataFrame, DataFrame containing MFCC features.
        """
        audio2data = self.audio2_segment_indexes()
        segment_2_mfccs = {}
        for audio, data in tqdm(audio2data.items()):
            processed_signal = (
                self.preprocess_audio_file(audio)
            )
            for segment_data in data:
                name, st, end = segment_data
                segment_signal = processed_signal[st:end]
                if len(segment_signal) == 0:
                    continue
                # Mel spectrogram
                mel_spec = (
                    self.audio_frame_to_mel_spectrogram(
                        segment_signal,
                        **self.kwargs
                    )
                )
                
                spec_path = os.path.join(self.path_save_spec_arrays,f"{name}.npy")
                np.save(spec_path, mel_spec)
                # MFCC extraction
                mfccs = self.segment_mfccs(segment_signal)
                segment_2_mfccs[name] = mfccs

        mfcc_names = np.array([f"x{i}" for i in range(40)])
        df = pd.DataFrame.from_dict(
            segment_2_mfccs,
            orient="index",
            columns=mfcc_names
        )
        df.index.name = "Segment"
        df.to_csv(
            "mfcc_data.csv",
            index=True
        )
        return df

In [23]:
p_odio = "../audio_files/"
p_anno = "../segment_index_extraction/segment_data.csv"
p_arra = "spectrogram_arrays/"
cutoff = 20000
nyq_fr = 22050

In [24]:
WINDOW_LENGTH= 1024 # size of the window when applying STFT
HOPE_LENGTH = 256   # number of samples to jump between two consecutive windows
NUMBER_MELS = 128  # mels bins, this will be the HEIGHT of the mel spectrogram
SAMPLE_RATE = 44100 # sample rate used to down-sample the audio file
POWER = 1            # power of order 1, order i.e. squared energy
F_MIN = 4000       # minimum frequency to be detected
F_MAX  = 8000        # maximum frequency to be detected

In [25]:
PARAMETERS = {"n_fft":WINDOW_LENGTH,
              "hop_length":HOPE_LENGTH,
              "n_mels":NUMBER_MELS,
              "sr":SAMPLE_RATE,
              "power":POWER,
              "fmin":F_MIN,
              "fmax":F_MAX
             }

In [26]:
cricket = SpectrogramAndMFCCExtraction(p_odio, p_anno, p_arra, cutoff, nyq_fr, **PARAMETERS)

In [27]:
df = cricket.Execution()

100%|█████████████████████████████████████████████| 7/7 [00:58<00:00,  8.39s/it]


In [28]:
# MFCC feature vectors
df

,x0,x1,x2,x3,x4,x5,x6,x7,x8,x9,...,x30,x31,x32,x33,x34,x35,x36,x37,x38,x39
Segment,,,,,,,,,,,,,,,,,,,,,
LPL11_0,-659.264451,11.433041,-100.840768,-32.852908,12.299486,7.033111,-0.316344,-0.023940,10.140916,-2.036533,...,13.697441,9.874818,8.620555,7.244674,5.567466,4.965056,5.450313,6.282713,4.106054,4.290943
LPL11_1,-652.958935,14.986218,-104.236728,-34.752288,18.363515,7.632606,-2.597473,0.335826,9.266835,-0.015383,...,13.954964,8.988994,9.627346,7.340720,5.411353,4.581652,5.535687,6.026212,4.867820,4.091970
LPL11_2,-648.262797,14.061155,-106.619528,-39.074585,16.749164,8.377479,-4.321320,-0.068532,10.070839,0.203587,...,14.574010,9.018942,10.309621,7.742263,5.629099,6.018623,6.079194,6.902125,4.730726,4.326731
LPL11_3,-653.016180,14.769442,-95.353215,-38.293455,14.895999,6.182915,-5.976756,-1.057661,10.868923,0.055778,...,14.560308,8.704203,9.364988,7.473805,5.489649,5.614038,5.791130,5.888839,4.659858,3.693330
LPL11_4,-643.733905,13.946340,-102.373741,-36.525467,21.630914,2.750695,-5.820327,2.391341,6.208309,0.609164,...,14.483075,9.412584,8.762501,7.558840,4.826981,5.897984,5.654952,5.365793,4.922365,3.846416
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
RBB13_355,-694.856208,10.374419,-100.305141,-25.545893,20.243351,2.237874,-4.241619,10.557901,-4.838256,-3.062559,...,11.126358,7.222105,9.071288,7.463955,6.145355,6.717863,4.206799,5.111120,4.745910,4.066091
RBB13_356,-687.987294,8.769893,-101.849774,-26.045941,20.779883,3.019842,-3.670531,10.667448,-5.438169,-2.152932,...,10.056862,7.925251,9.179686,6.701276,6.101635,6.568662,5.601872,5.240036,4.122834,4.294362
RBB13_357,-692.709742,9.503814,-96.641140,-29.618889,19.882132,4.145899,-9.315718,12.062717,-2.855589,-3.577344,...,11.581924,7.696684,9.727999,6.521906,6.168448,6.143742,6.392934,5.513157,4.454830,4.278504


In [29]:
# Display mel spectrogram examples
imgs = os.listdir(p_arra)
img = np.load(p_arra+imgs[2])
plt.imshow(img)

NameError: name 'plt' is not defined